# Pipeline RAG con OpenAI + Pinecone

Este notebook demuestra un pipeline de **Retrieval-Augmented Generation (RAG)** usando:
- **OpenAI** – `text-embedding-3-small` para embeddings y `gpt-4o-mini` como LLM de chat.
- **Pinecone** – base de datos vectorial para almacenar y recuperar embeddings de documentos.
- **LangChain v0.2+** – orquestación mediante LCEL (LangChain Expression Language).

## Vista general del pipeline

```
Archivos TXT  →  TextLoader  →  RecursiveCharacterTextSplitter
                                        ↓
                            OpenAIEmbeddings (text-embedding-3-small)
                                        ↓
                            PineconeVectorStore (upsert)
                                        ↓
                        VectorStoreRetriever (búsqueda top-k por similitud)
                                        ↓
                    ChatPromptTemplate + ChatOpenAI (gpt-4o-mini)
                                        ↓
                                  Respuesta final
```

## Prerrequisitos

1. Instala dependencias: `pip install -r requirements.txt`
2. Copia `.env.example` a `.env` y completa tus API keys.
3. Crea un índice en Pinecone con **dimension=1536** y **metric=cosine** (ver Paso 4 más abajo).

## Paso 1 – Cargar variables de entorno

Usamos `python-dotenv` para cargar secretos desde un archivo `.env`.  
Copia `.env.example` → `.env` y completa tus credenciales antes de ejecutar esta celda.

In [13]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Load .env from the project root (one level above notebooks/)
env_path = Path("../.env")
if env_path.exists():
    load_dotenv(dotenv_path=env_path, override=True)
    print(f"✅  Loaded environment variables from {env_path.resolve()}")
else:
    load_dotenv(override=True)  # fall back to environment variables already set in the shell
    print("ℹ️  No .env file found – using shell environment variables.")

# Verify required variables are present
required_vars = ["OPENAI_API_KEY", "PINECONE_API_KEY", "PINECONE_INDEX_NAME"]
missing = [v for v in required_vars if not os.getenv(v)]
if missing:
    raise EnvironmentError(f"Missing required environment variables: {missing}")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"].strip().strip("\"'")
PINECONE_API_KEY = os.environ["PINECONE_API_KEY"].strip().strip("\"'")
INDEX_NAME = os.environ["PINECONE_INDEX_NAME"].strip()
NAMESPACE = os.getenv("PINECONE_NAMESPACE", "default").strip()

if not PINECONE_API_KEY.startswith("pcsk_"):
    print("⚠️  PINECONE_API_KEY does not start with 'pcsk_'. Verify your key in .env.")

print(f"📌  Pinecone index : {INDEX_NAME}")
print(f"📌  Namespace      : {NAMESPACE}")
print(f"🔑  Pinecone key    : {PINECONE_API_KEY[:8]}... (masked)")

✅  Loaded environment variables from C:\Users\USER\Documents\Octavo Semestre\TDSE\Rag-Project-Openai-Pinecone\.env
📌  Pinecone index : rag-openai-1536
📌  Namespace      : default
🔑  Pinecone key    : pcsk_6DZ... (masked)


## Paso 2 – Cargar documentos TXT

Usamos `DirectoryLoader` + `TextLoader` de LangChain para cargar todos los `.txt` dentro de `data/`.  
Cada archivo se convierte en un `Document` con `page_content` (texto crudo) y `metadata` (incluyendo la ruta `source`).

In [2]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path("..")))

from src.rag_utils import load_txt_documents

DATA_DIR = "../data"

documents = load_txt_documents(DATA_DIR)

# Preview the first document
print("\n--- First document preview ---")
print(f"Source : {documents[0].metadata['source']}")
print(f"Length : {len(documents[0].page_content)} characters")
print("\nFirst 300 characters:")
print(documents[0].page_content[:300], "...")

c:\Users\USER\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 3/3 [00:00<00:00, 111.32it/s]

Loaded 3 document(s) from '../data'.

--- First document preview ---
Source : ..\data\ai_overview.txt
Length : 1560 characters

First 300 characters:
Artificial Intelligence (AI) is the simulation of human intelligence processes by machines, especially computer systems.
These processes include learning (the acquisition of information and rules for using the information), reasoning (using rules
to reach approximate or definite conclusions), and se ...


## Paso 3 – Dividir documentos en chunks

`RecursiveCharacterTextSplitter` divide documentos largos en fragmentos más pequeños con solapamiento.  
El solapamiento evita perder contexto en los límites entre chunks.

| Parámetro        | Valor | Significado                                      |
|------------------|-------|--------------------------------------------------|
| `chunk_size`     | 1000  | Máximo de caracteres por chunk                  |
| `chunk_overlap`  | 200   | Caracteres compartidos entre chunks consecutivos |

In [3]:
from src.rag_utils import split_documents

CHUNK_SIZE    = 1000
CHUNK_OVERLAP = 200

chunks = split_documents(documents, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

print(f"\nTotal chunks : {len(chunks)}")
print("\n--- Sample chunk ---")
print(chunks[0].page_content)
print("\nMetadata:", chunks[0].metadata)

Split into 7 chunk(s) (chunk_size=1000, overlap=200).

Total chunks : 7

--- Sample chunk ---
Artificial Intelligence (AI) is the simulation of human intelligence processes by machines, especially computer systems.
These processes include learning (the acquisition of information and rules for using the information), reasoning (using rules
to reach approximate or definite conclusions), and self-correction. AI has evolved dramatically since its inception in the
1950s and is now embedded in many everyday technologies.

AI can be categorized into two main types: Narrow AI (also called Weak AI), which is designed to perform a specific task
such as voice recognition or image classification, and General AI (also called Strong AI), which would perform any
intellectual task that a human can do. Currently, all practical AI systems are Narrow AI.

Metadata: {'source': '..\\data\\ai_overview.txt'}


## Paso 4 – Crear / verificar el índice de Pinecone

El modelo de embeddings `text-embedding-3-small` produce vectores de **1536 dimensiones**.  
Tu índice de Pinecone debe crearse con esa dimensión **antes** de ejecutar el upsert.

### Opción A – Crear desde la consola de Pinecone (recomendado)
1. Ve a [https://app.pinecone.io](https://app.pinecone.io).
2. Haz clic en **Create index**.
3. Define **Name** = valor de `PINECONE_INDEX_NAME`, **Dimensions** = `1536`, **Metric** = `cosine`.
4. Haz clic en **Create index** y espera hasta que el estado sea **Ready**.

### Opción B – Crear programáticamente (ejecuta la celda de abajo)

In [14]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

try:
    existing_indexes = [idx.name for idx in pc.list_indexes()]
except Exception as exc:
    msg = str(exc)
    if "Unauthorized" in msg or "Invalid API Key" in msg or "401" in msg:
        raise RuntimeError(
            "Pinecone authentication failed (401 Unauthorized). "
            "Please verify PINECONE_API_KEY in your .env file and ensure it belongs to the same Pinecone project."
        ) from exc
    raise

print(f"Existing indexes: {existing_indexes}")

if INDEX_NAME not in existing_indexes:
    print(f"Creating index '{INDEX_NAME}' ...")
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,        # must match text-embedding-3-small output dimension
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    # Wait until the index is ready
    import time
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        print("  Waiting for index to be ready...")
        time.sleep(5)
    print(f"✅  Index '{INDEX_NAME}' is ready.")
else:
    print(f"✅  Index '{INDEX_NAME}' already exists.")

Existing indexes: ['rag-openai-1536', 'rag-ollama-768']
✅  Index 'rag-openai-1536' already exists.


## Paso 5 – Generar embeddings y hacer upsert en Pinecone

Este paso:
1. Crea una instancia de `OpenAIEmbeddings` usando `text-embedding-3-small`.
2. Genera embeddings de cada chunk y sube los vectores al índice de Pinecone.

> **Nota:** Si ya hiciste upsert de documentos antes y quieres evitar re-ingesta, salta directamente al **Paso 6**.

In [15]:
from src.rag_utils import build_vectorstore

EMBEDDING_MODEL = "text-embedding-3-small"

vectorstore = build_vectorstore(
    chunks=chunks,
    index_name=INDEX_NAME,
    namespace=NAMESPACE,
    embedding_model=EMBEDDING_MODEL,
)

print("\n✅  Vectors upserted successfully.")

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

## Paso 6 – (Opcional) Conectar a un vectorstore existente

Si ya hiciste upsert de tus documentos en una ejecución previa, usa esta celda en lugar del Paso 5  
para conectarte al índice existente sin volver a generar embeddings.

In [16]:
from src.rag_utils import load_vectorstore

EMBEDDING_MODEL = "text-embedding-3-small"
vectorstore = load_vectorstore(
    index_name=INDEX_NAME,
    namespace=NAMESPACE,
    embedding_model=EMBEDDING_MODEL,
)
print("✅  Connected to existing vectorstore.")

✅  Connected to existing vectorstore.


## Paso 7 – Construir la cadena RAG

Conectamos todo usando **LCEL** (LangChain Expression Language):

```
pregunta
   │
   ├─► retriever  ──► format_docs ──► contexto
   │                                      │
   └──────────────────────────────► ChatPromptTemplate
                                          │
                                    ChatOpenAI (gpt-4o-mini)
                                          │
                                    StrOutputParser
                                          │
                                       respuesta (str)
```

In [17]:
from src.rag_utils import build_rag_chain

LLM_MODEL   = "gpt-4o-mini"
TOP_K       = 4          # number of chunks to retrieve per query
TEMPERATURE = 0.0        # deterministic output

rag_chain, retriever = build_rag_chain(
    vectorstore=vectorstore,
    top_k=TOP_K,
    llm_model=LLM_MODEL,
    temperature=TEMPERATURE,
)

print("✅  RAG chain built.")
print(f"    LLM        : {LLM_MODEL}")
print(f"    Retrieval  : top-{TOP_K} chunks")

✅  RAG chain built.
    LLM        : gpt-4o-mini
    Retrieval  : top-4 chunks


## Paso 8 – Ejecutar consultas RAG

¡Haz preguntas! El pipeline recuperará los chunks más relevantes desde Pinecone  
y los pasará como contexto al LLM para generar una respuesta fundamentada.

In [19]:
def ask(question: str) -> str:
    """Run a RAG query and print the answer."""
    print(f"\n❓ Question: {question}")
    answer = rag_chain.invoke(question)
    print(f"\n💬 Answer:\n{answer}")
    return answer

# Example queries
ask("What is Artificial Intelligence and what are its main application areas?")


❓ Question: What is Artificial Intelligence and what are its main application areas?


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
ask("What are the main types of machine learning? Give a brief description of each.")

In [ ]:
ask("What is Retrieval-Augmented Generation (RAG) and why is it useful?")

## Paso 9 – Inspeccionar fuentes recuperadas

Podemos invocar el retriever directamente para ver qué chunks se recuperan para una consulta dada,  
lo cual permite verificar que se está trayendo el contexto correcto.

In [21]:
query = "What are the limitations of Large Language Models?"

retrieved_docs = retriever.invoke(query)

print(f"🔍  Retrieved {len(retrieved_docs)} chunk(s) for: \"{query}\"\n")
for i, doc in enumerate(retrieved_docs, start=1):
    source = doc.metadata.get("source", "unknown")
    print(f"── Chunk {i} ── source: {source}")
    print(doc.page_content[:300], "...\n")

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [22]:
# Ask the full RAG question with sources displayed afterwards
question = "What are the limitations of Large Language Models?"
answer = ask(question)

print("\n📚  Sources used:")
for doc in retriever.invoke(question):
    print(" -", doc.metadata.get("source", "unknown"))


❓ Question: What are the limitations of Large Language Models?


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

## Resumen

En este notebook:

| Paso | Acción |
|------|--------|
| 1 | Cargamos las API keys desde `.env` |
| 2 | Cargamos documentos TXT desde `data/` |
| 3 | Dividimos documentos en chunks con solapamiento |
| 4 | Verificamos / creamos el índice de Pinecone |
| 5 | Generamos embeddings y subimos chunks a Pinecone |
| 6 | *(Opcional)* Conectamos a un vectorstore existente |
| 7 | Construimos la cadena RAG con LCEL |
| 8 | Ejecutamos consultas de ejemplo y revisamos respuestas |
| 9 | Inspeccionamos documentos fuente recuperados |

### Cierre argumentativo: limitantes y cumplimiento

**Sobre los limitantes:**
- El principal bloqueo observado fue `RateLimitError: 429 (insufficient_quota)` desde OpenAI en tareas de embeddings y consultas.
- Este error corresponde a disponibilidad/cuota de API y no a un error de diseño del pipeline RAG ni de la integración con Pinecone.
- También se validó que el acceso a Pinecone funciona correctamente una vez configurada una key válida.

**Sobre lo requerido en el laboratorio:**
- Se implementó el flujo completo de RAG solicitado: carga, chunking, indexación vectorial, recuperación y generación.
- Se estructuró el proyecto en funciones reutilizables (`src/rag_utils.py`) y en un notebook guiado por pasos.
- Se documentó una ruta operativa para continuar cuando hay límite de cuota: conectar a vectorstore existente (Paso 6) o usar un stack alternativo con Llama/Ollama.

### Próximos pasos

- Agrega más archivos TXT a `data/` y vuelve a ejecutar desde el **Paso 2**.
- Experimenta con `chunk_size`, `chunk_overlap` y `top_k`.
- Cambia el modelo LLM (`gpt-4o`, `gpt-3.5-turbo`) o el modelo de embeddings.
- Usa diferentes namespaces en Pinecone para aislar conjuntos de documentos.

### Conclusión académica

En conclusión, el desarrollo presentado cumple con los objetivos técnicos del laboratorio al implementar y validar un pipeline RAG completo con separación modular, trazabilidad por etapas y evidencias de recuperación contextual; además, los incidentes observados durante la ejecución se atribuyen a restricciones externas de cuota en servicios API y no a defectos de arquitectura, por lo que la solución se considera metodológicamente correcta, reproducible y adaptable mediante alternativas de despliegue como el uso de vectorstores ya poblados o modelos locales tipo Llama/Ollama.